In [4]:
import json
import os
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.deepsearchqa_grader import (
    EvaluationOutcome,
    evaluate_deepsearchqa_async,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from aieng.agent_evals.langfuse import upload_dataset_to_langfuse
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from langfuse.experiment import Evaluation
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

DATASET_NAME = "DeepSearchQA-Sun-Life"

2026-03-24 14:47:12,049 INFO dotenv.main: python-dotenv could not find configuration file .env.


Working directory set to: /


In [5]:
dataset = DeepSearchQADataset()
examples = dataset.get_by_category("Finance & Economics")[:10]

console.print(f"Uploading [cyan]{len(examples)}[/cyan] examples to dataset '{DATASET_NAME}'...")

# Write examples to a temporary JSONL file for the upload utility
with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False, encoding="utf-8") as f:
    for ex in examples:
        record = {
            "input": ex.problem,
            "expected_output": ex.answer,
            "metadata": {
                "example_id": ex.example_id,
                "category": ex.problem_category,
                "answer_type": ex.answer_type,
            },
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    temp_path = f.name

await upload_dataset_to_langfuse(dataset_path=temp_path, dataset_name=DATASET_NAME)
os.unlink(temp_path)

console.print(f"[green]✓[/green] Dataset '{DATASET_NAME}' ready in Langfuse")

2026-03-24 14:47:31,173 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Downloading DeepSearchQA dataset...
2026-03-24 14:47:32,373 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Dropped 4 examples with missing answers
2026-03-24 14:47:32,374 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Loaded 896 examples from DeepSearchQA


Uploading 10 examples to dataset 'DeepSearchQA-Sun-Life'...

2026-03-24 14:47:32,668 INFO aieng.agent_evals.langfuse: Loading dataset from '/tmp/tmp288wi9r_.jsonl'


Uploading Langfuse dataset 'DeepSearchQA-Sun-Life' ━━━━━━━ 10/10 0:00:00 0:00:0000:0000:00


2026-03-24 14:47:33,726 INFO aieng.agent_evals.langfuse: Uploaded 10 items to dataset 'DeepSearchQA-Sun-Life'


✓ Dataset 'DeepSearchQA-Sun-Life' ready in Langfuse